In [ ]:
# check for valid gpu
!nvidia-smi

In [ ]:
# establish home directory for project
import os
home = os.getcwd()
home

In [ ]:
# pandas to write results to csv
import pandas as pd
# opencv for frame processing
import cv2
# check ultralytics version
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

In [ ]:
# load pre-trained model
model = YOLO(f"{home}/runs/detect/train3/weights/best.pt")

In [ ]:
# class name values
model.names

### Python per-frame inference

In [ ]:
# for every video in source directory, run detection and save results to csv
for video in os.listdir(f"{home}/videos"):
    if video.endswith(".mp4"):
        # set source to video filepath
        source = (f"{home}/videos/{video}")

        # capture video for detection
        cap = cv2.VideoCapture(source)

        # empty list to store box data
        data = []
        # frame counter to store for each box
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1

            # track objects in the current frame
            results = model.track(frame, persist=True)

            # for each box found in each frame, store the box data
            for result in results:
                if result.boxes is not None and len(result.boxes) > 0:
                    for i in range(len(result.boxes)):
                        data.append({
                            # if not none else none handles missing values, while int() and float() convert from tensor to numbers
                            'frame': frame_count,
                            'id': int(result.boxes.id[i]) if result.boxes.id is not None else None,
                            'class': model.names[int(result.boxes.cls[i]) if result.boxes.cls is not None else None], # sets class to nominal name
                            'confidence': float(result.boxes.conf[i]) if result.boxes.conf is not None else None,
                            'x1': int(result.boxes.xyxy[i][0]) if result.boxes.xyxy[i][0] is not None else None,
                            'y1': int(result.boxes.xyxy[i][1]) if result.boxes.xyxy[i][1] is not None else None,
                            'x2': int(result.boxes.xyxy[i][2]) if result.boxes.xyxy[i][2] is not None else None,
                            'y2': int(result.boxes.xyxy[i][3]) if result.boxes.xyxy[i][3] is not None else None
                        })

        # create dataframe from box data list
        df = pd.DataFrame(data)
        # save results to csv
        df.to_csv(f"{home}/results/{video}_results.csv", index=False, header=True)
    else:
        print(f"{video} is not a valid video file (.mp4)")

### Python per-frame inference with no feeder

In [ ]:
# for every video in directory, run detection and save results to csv
for video in os.listdir(f"{home}/videos"):
    if video.endswith(".mp4"):
        # set source to video filepath
        source = (f"{home}/videos/{video}")

        # capture video for detection
        cap = cv2.VideoCapture(source)

        # empty list to store box data
        data = []
        # frame counter to store for each box
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1

            # track objects in the current frame
            results = model.track(frame, persist=True,classes=[0,2,3,4,5,6]) # omit class 1 (feeder) from tracking

            # for each box found in each frame, store the box data
            for result in results:
                if result.boxes is not None and len(result.boxes) > 0:
                    for i in range(len(result.boxes)):
                        data.append({
                            # if not none else none handles missing values, while int() and float() convert from tensor to numbers
                            'frame': frame_count,
                            'id': int(result.boxes.id[i]) if result.boxes.id is not None else None,
                            'class': model.names[int(result.boxes.cls[i]) if result.boxes.cls is not None else None], # sets class to nominal name
                            'confidence': float(result.boxes.conf[i]) if result.boxes.conf is not None else None,
                            'x1': int(result.boxes.xyxy[i][0]) if result.boxes.xyxy[i][0] is not None else None,
                            'y1': int(result.boxes.xyxy[i][1]) if result.boxes.xyxy[i][1] is not None else None,
                            'x2': int(result.boxes.xyxy[i][2]) if result.boxes.xyxy[i][2] is not None else None,
                            'y2': int(result.boxes.xyxy[i][3]) if result.boxes.xyxy[i][3] is not None else None
                        })

        # create dataframe from box data list
        df = pd.DataFrame(data)
        # save results to csv
        df.to_csv(f"{home}/results/nf_{video}_results.csv", index=False, header=True)
    else:
        print(f"{video} is not a valid video file (.mp4)")